# U5 — Calculus for ML: Lab

In [ ]:

import sympy as sp

x, y = sp.symbols('x y')
sp.init_printing()
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.0.2


#1. Derivatives — the slope of a function

In [ ]:
 NUMERICAL DERIVATIVE (finite difference)
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')

Numerical f'(3): 6.0  (exact = 6)


In [ ]:
SYMBOLIC DERIVATIVE

expr = x**2
deriv = sp.diff(expr, x)
print('d/dx (x**2) =', deriv)
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


LAB EXERCISE 1 — Numerical vs symbolic derivative

For the function `g(x) = x**3 + 2*x`:
1. Compute the **numerical** derivative at `x = 2` using a finite difference.
2. Compute the **symbolic** derivative with `sp.diff` and print it.
3. Evaluate the symbolic derivative at `x = 2` and confirm it matches step 1.

In [ ]:

import sympy as sp

def g(x):
    return x**3 + 2*x

h = 1e-6
x0 = 2
numerical_derivative = (g(x0 + h) - g(x0)) / h
print("Numerical derivative at x = 2:", numerical_derivative)

x = sp.Symbol('x')
g_sym = x**3 + 2*x
symbolic_derivative = sp.diff(g_sym, x)
print("Symbolic derivative:", symbolic_derivative)

symbolic_value = symbolic_derivative.subs(x, 2)
print("Symbolic derivative at x = 2:", symbolic_value)

print("Difference:", abs(numerical_derivative - float(symbolic_value)))

Numerical derivative at x = 2: 14.000006002490295
Symbolic derivative: 3*x**2 + 2
Symbolic derivative at x = 2: 14
Difference: 6.002490295031748e-06


#2. Partial derivatives & the gradient

In [ ]:
 PARTIAL DERIVATIVE

f2 = x**2 + 3*x*y + y**2

print('df/dx =', sp.diff(f2, x))
print('df/dy =', sp.diff(f2, y))

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [ ]:
 THE GRADIENT (vector of partials)

grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)


grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


 LAB EXERCISE 2 — Compute a gradient

For `h2 = x**2 * y + sp.sin(y)`:
1. Compute `dh/dx` and `dh/dy` with `sp.diff`.
2. Assemble the gradient as a list `[dh/dx, dh/dy]`.
3. Evaluate the gradient at the point `(x=2, y=0)`.

In [ ]:
h2 = x**2 * y + sp.sin(y)
h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
dh_dx = sp.diff(h2, x)
dh_dy = sp.diff(h2, y)

print("dh/dx =", dh_dx)
print("dh/dy =", dh_dy)

# 2. Assemble the gradient list
gradient = [dh_dx, dh_dy]
print("Gradient =", gradient)

# 3. Evaluate at (x=2, y=0)
gradient_at_point = [expr.subs({x: 2, y: 0}) for expr in gradient]
print("Gradient at (2, 0) =", gradient_at_point)

dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
Gradient = [2*x*y, x**2 + cos(y)]
Gradient at (2, 0) = [0, 5]


#3. The chain rule

In [ ]:

by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

In [ ]:

expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))

LAB EXERCISE 3 — Chain rule

For `y = sp.exp(x**2 + 1)`:
1. Write the chain-rule result **by hand** as a SymPy expression (outer derivative × inner derivative).
2. Differentiate the same expression with `sp.diff`.
3. Confirm the two match using `sp.simplify(by_hand - by_sympy) == 0`.

In [ ]:


# 1. By hand (as a SymPy expression)
dy_dx_hand = sp.exp(x**2 + 1) * 2*x
print("By hand:", dy_dx_hand)

# 2. With sp.diff
y = sp.exp(x**2 + 1)
dy_dx_diff = sp.diff(y, x)
print("With sp.diff:", dy_dx_diff)

# 3. Confirm they match
print("Match:", sp.simplify(dy_dx_hand - dy_dx_diff) == 0)

By hand: 2*x*exp(x**2 + 1)
With sp.diff: 2*x*exp(x**2 + 1)
Match: True


#4. Backpropagation — the chain rule in a network

A 2-layer network is just a composition:  **x → (W1) → ReLU → (W2) → y_hat**. Backprop applies the chain rule from the loss backwards to every weight.

In [ ]:

X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1
h  = np.maximum(0, z1)
y_hat = h @ W2
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

Initial loss: 1.6936


In [ ]:

dy   = 2 * (y_hat - Y) / Y.size
dW2  = h.T @ dy
dh   = dy @ W2.T
dz1  = dh * (z1 > 0)
dW1  = X.T @ dz1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [ ]:


lr = 0.1
W1 -= lr * dW1
W2 -= lr *
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Loss before: 1.6936
Loss after : 1.6524 -> should be lower


LAB EXERCISE 4 — Derive the gradients for a 2-layer network

Fresh weights are set up below. Using the chain rule (reuse the pattern from 4A/4B):
1. Run the **forward pass**: compute `z1`, `h` (ReLU), `y_hat`, and `loss`.
2. Run the **backward pass**: compute `dy`, `dW2`, `dh`, `dz1`, `dW1`.
3. Take one gradient-descent step (`lr = 0.05`) and print the loss before and after.

In [ ]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1
Wb = np.random.randn(8, 1) * 0.1


Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1
Wb = np.random.randn(8, 1) * 0.1

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss = np.mean((y_hat - Yb) ** 2)

# 2. Backward pass: dy, dWb, dh, dz1, dWa
dy = 2 * (y_hat - Yb) / Yb.shape[0]
dWb = h.T @ dy
dh = dy @ Wb.T
dz1 = dh * (z1 > 0)
dWa = Xb.T @ dz1

# 3. One step with lr = 0.05; print loss before and after
lr = 0.05

print("Loss before:", loss)
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1
Wb = np.random.randn(8, 1) * 0.1

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss = np.mean((y_hat - Yb) ** 2)

# 2. Backward pass: dy, dWb, dh, dz1, dWa
dy = 2 * (y_hat - Yb) / Yb.shape[0]
dWb = h.T @ dy
dh = dy @ Wb.T
dz1 = dh * (z1 > 0)
dWa = Xb.T @ dz1

# 3. One step with lr = 0.05; print loss before and after
lr = 0.05

print("Loss before:", loss)

Wa -= lr * dWa
Wb -= lr * dWb

# Calculate loss after update
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss_after = np.mean((y_hat - Yb) ** 2)

print("Loss after:", loss_after)
Wa -= lr * dWa
Wb -= lr * dWb

# Calculate loss after update
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss_after = np.mean((y_hat - Yb) ** 2)

print("Loss after:", loss_after)




Loss before: 1.1706051565963387
Loss before: 0.9012858906993045
Loss after: 0.8998731831845102
Loss after: 0.8984384794675458


#5. Hessian & gradient descent

In [1]:


f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)

NameError: name 'x' is not defined

In [ ]:

xv = 0.0
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)
    xv = xv - lr * grad
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

SyntaxError: invalid syntax (2523844755.py, line 1)

 LAB EXERCISE 5 — Hessian + gradient descent

1. Compute the **Hessian** of `f = x**4 + y**2` with `sp.hessian`.
2. Run **gradient descent** to minimise `f(x) = (x - 7)**2`:
   start at `x = 0`, `lr = 0.1`, 20 steps, using `f'(x) = 2*(x - 7)`. Print the final `x` (should approach 7).

In [ ]:

x, y = sp.symbols('x y')

f = x**4 + y**2

H = sp.hessian(f, (x, y))
print("Hessian:")
print(H)

# 2. Gradient descent to minimise (x - 7)**2
xv = 0.0
lr = 0.1

for step in range(20):
    grad = 2 * (xv - 7)
    xv = xv - lr * grad

print('Final x:', xv)

Hessian:
Matrix([[12*x**2, 0], [0, 2]])
Final x: 6.91929549467752
